In [11]:
#mount google drive
from google.colab import drive
import os
drive.mount('/content/drive')
DATA_PATH  = os.path.join(os.getcwd(),'drive', 'MyDrive', 'DSE_260A', 'data')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#If running locally instead of colab
DATA_PATH = '../data/'

In [3]:
# Import required libraries
import os
import pandas as pd

In [6]:
# Load the dataset
df = pd.read_csv(os.path.join(DATA_PATH, 'challenge_transcriptomics.tsv'), sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2024_UGA.ID_128,0,ENSG00000000003,6,0.287,Unknown
1,2024_UGA.ID_128,0,ENSG00000000005,0,0.000,Unknown
2,2024_UGA.ID_128,0,ENSG00000000419,442,10.325,Unknown
3,2024_UGA.ID_128,0,ENSG00000000457,245,7.713,Unknown
4,2024_UGA.ID_128,0,ENSG00000000460,102,3.702,Unknown


In [7]:
# raw_count has actual values; material is uniformly 'Unknown'
# Drop raw_count to keep tpm_count only, consistent with training pipeline
# Drop material (constant 'Unknown')
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: ['Unknown']
timepoints: [np.int64(0), np.int64(7)]
participants: 40


In [8]:
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2024_UGA.ID_128,0,ENSG00000000003,0.287
1,2024_UGA.ID_128,0,ENSG00000000005,0.000
2,2024_UGA.ID_128,0,ENSG00000000419,10.325
3,2024_UGA.ID_128,0,ENSG00000000457,7.713
4,2024_UGA.ID_128,0,ENSG00000000460,3.702


In [9]:
# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "TRAN2020_{gene}_d{timepoint}" format
df_pivot.columns = [f'TRAN_{gene}_d{int(tp)}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)

(40, 109805)


In [13]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_parquet(os.path.join(DATA_PATH,'cleaned_data', 'challenge_transcriptomics_cleaned.parquet'), index=False)